# Verification of manuscript numbers

Loads the project data and confirms (or refutes) the numbers quoted in the text.
Run on the HPC, where `CMIG_DATA` and the repo `data/` directory are populated.

Claims checked:

1. *We then perform a truncated PCA via singular value decomposition, retaining the
   first 15 modes, which explain XX% of the total ocean variance.*
2. *In nearly all XX river basins and YY ensemble members the correlation is positive,
   with a mean correlation of 0.27 (Figure 5a), whereas the null distribution is
   centered on zero.*
3. *Stability of the SST sensitivity over time across basins and ensemble members that
   had data from at least 1980-2024* -- and the number of 30-year windows.

Each section prints the numbers to substitute, plus any discrepancy between what the
text says and what the code actually does.

In [ ]:
import importlib.util
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

# --- locate project root (directory containing paths.py) ---
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'paths.py').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'paths.py').exists():
    raise RuntimeError('Could not find project root containing paths.py')

sys.path.insert(0, str(PROJECT_ROOT))
from paths import CMIG_DATA, DATA_DIR

SCRIPTS_DIR = PROJECT_ROOT / 'code' / 'scripts'
sys.path.insert(0, str(SCRIPTS_DIR))

INPUTS_DIR = CMIG_DATA / 'fperlmutter/Observational_Regressions_Project/Data/Processed'

warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)

print('PROJECT_ROOT :', PROJECT_ROOT)
print('INPUTS_DIR   :', INPUTS_DIR, '  exists:', INPUTS_DIR.exists())
print('DATA_DIR     :', DATA_DIR, '  exists:', DATA_DIR.exists())

In [ ]:
# Import 17_PCA_method.py by path (module name starts with a digit, so no plain import).
# Using the project's own functions means this notebook measures the real pipeline,
# not a re-implementation of it. main() is guarded, so nothing runs on import.

def load_module_by_path(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

pca = load_module_by_path('pca_method', SCRIPTS_DIR / '17_PCA_method.py')

print('K_MAX      :', pca.K_MAX)
print('K_FALLBACK :', pca.K_FALLBACK)
print('ALPHA      :', pca.ALPHA)
print('MIN_YEARS  :', pca.MIN_YEARS)

---
## 1. Truncated PCA: variance explained by the first 15 modes

`compute_pca_truncated` returns `var_exp = s**2 / sum(X_w**2)`, i.e. the fraction of
**area-weighted** (sqrt-cos-latitude) variance of the **detrended, ocean-only** SST
anomaly field. Ocean cells are those finite at every time step.

Computed separately for each SST dataset, since the number is grid- and
record-dependent.

In [ ]:
K_REPORT = 15
SST_DATASETS = ['ERSSTv6', 'COBE-SST3']

var_tables = {}

for sst_name in SST_DATASETS:
    path = INPUTS_DIR / f'sst_anom_{sst_name}.nc'
    if not path.exists():
        print(f'[missing] {path}')
        continue

    sst = xr.open_dataarray(path).squeeze()
    if 'lev' in sst.dims:
        sst = sst.squeeze('lev', drop=True)
    sst.load()

    flat, w_flat, ocean_mask = pca.flatten_sst(sst)
    flat_det = pca.detrend_along_time(flat)
    _, _, var_exp = pca.compute_pca_truncated(flat_det, w_flat, K_REPORT)

    cum = np.cumsum(var_exp)
    var_tables[sst_name] = pd.DataFrame({
        'mode': np.arange(1, K_REPORT + 1),
        'var_explained_pct': var_exp * 100,
        'cumulative_pct': cum * 100,
    }).set_index('mode')

    t0 = str(sst['time'].values[0])[:7]
    t1 = str(sst['time'].values[-1])[:7]
    print(f'--- {sst_name} ---')
    print(f'  record      : {t0} to {t1}  ({sst.sizes["time"]} months)')
    print(f'  ocean cells : {flat.shape[1]:,} of {ocean_mask.size:,}')
    print(f'  mode 1      : {var_exp[0] * 100:.1f}%')
    print(f'  modes 1-5   : {cum[4] * 100:.1f}%')
    print(f'  modes 1-15  : {cum[14] * 100:.1f}%   <-- XX for claim 1')
    print()

    del sst, flat, flat_det

for name, tbl in var_tables.items():
    print(f'=== {name} ===')
    print(tbl.round(2).to_string())
    print()

### 1b. Caveat: the pipeline does not retain 15 modes

`K_MAX = 15` is the **maximum k tested in cross-validation**, not the number retained.
The final per-basin model uses `optimal_k`, chosen by the 1-SE rule
(`17_PCA_method.py:400-416`), with `K_FALLBACK = 4` when CV fails. So the sentence as
written describes something the code does not do, unless `optimal_k == 15` everywhere.

The cell below reports the actual distribution of `optimal_k` and of
`variance_explained_sum` (which is the variance at `optimal_k`, **not** at 15).

In [ ]:
pca_files = sorted(DATA_DIR.glob('global_pca_regression_*.nc'))
print(f'Found {len(pca_files)} PCA regression output files\n')

rows = []
for f in pca_files:
    ds = xr.open_dataset(f)
    k = ds['optimal_k'].values
    v = ds['variance_explained_sum'].values
    k_f = k[np.isfinite(k)]   # drop skipped basins so pct_k_eq_15 is not diluted by NaN
    if k_f.size == 0:
        ds.close()
        continue
    rows.append({
        'file': f.name.replace('global_pca_regression_', '').replace('.nc', ''),
        'n_basins': int(k_f.size),
        'k_median': float(np.median(k_f)),
        'k_mean': float(np.mean(k_f)),
        'k_min': float(np.min(k_f)),
        'k_max': float(np.max(k_f)),
        'pct_k_eq_15': float(np.mean(k_f == 15) * 100),
        'var_expl_mean_pct': float(np.nanmean(v) * 100),
    })
    ds.close()

if rows:
    k_table = pd.DataFrame(rows).set_index('file')
    print(k_table.round(2).to_string())
    print()
    print(f'Across all files: median optimal_k = {k_table["k_median"].median():.1f}, '
          f'{k_table["pct_k_eq_15"].mean():.1f}% of basins select k = 15')
    print(f'Mean variance explained at optimal_k = {k_table["var_expl_mean_pct"].mean():.1f}%')
else:
    print('No global_pca_regression_*.nc files found -- run 11_pca.sh first.')

---
## 2. Figure 5a: randomization experiment

Source files: `randomization_experiment_{P}_{SST}.nc` (from
`14_Linear_Regression_Randomization.py`).

- `original_correlation` -- dims `(basin,)`, one value per basin per ensemble member
- `bootstrap_correlations` -- dims `(bootstrap, basin)`, the null (SST block-resampled,
  precipitation held fixed)

An *ensemble member* here is one (precip x SST) pair, so YY is the number of files
found (8 precip x 2 SST = 16 if all are present). `Figure_05` pools all basins and all
members into one distribution, so the pooled mean below is exactly the number the
figure's dashed line marks.

In [ ]:
PRECIP_DATASETS = ['GPCP', 'CRU', 'GPCC', 'CPC', 'UDel', 'PREC', 'TerraClimate', 'REGEN']

member_rows = []
orig_pooled, null_pooled = [], []
orig_raw = {}          # unfiltered per-member arrays, for the all-members-positive check
basin_counts = {}

for p_name in PRECIP_DATASETS:
    for sst_name in SST_DATASETS:
        f = DATA_DIR / f'randomization_experiment_{p_name}_{sst_name}.nc'
        if not f.exists():
            print(f'[missing] {f.name}')
            continue

        ds = xr.open_dataset(f)
        orig = ds['original_correlation'].values.astype(float)      # (basin,)
        boot = ds['bootstrap_correlations'].values.astype(float)    # (bootstrap, basin)

        member = f'{p_name}_{sst_name}'
        basin_counts[member] = int(ds.sizes['basin'])
        orig_raw[member] = orig

        finite = np.isfinite(orig)
        # empirical one-sided p per basin: how often does the null match or beat the observed
        p_emp = np.full(orig.shape, np.nan)
        for b in range(orig.size):
            col = boot[:, b]
            col = col[np.isfinite(col)]
            if col.size and np.isfinite(orig[b]):
                p_emp[b] = np.mean(col >= orig[b])

        member_rows.append({
            'member': member,
            'n_basins': int(finite.sum()),
            'mean_corr': float(np.nanmean(orig)),
            'pct_positive': float(np.mean(orig[finite] > 0) * 100),
            'null_mean': float(np.nanmean(boot)),
            'pct_p_lt_05': float(np.mean(p_emp[np.isfinite(p_emp)] < 0.05) * 100),
        })

        orig_pooled.append(orig[finite])
        null_pooled.append(boot[np.isfinite(boot)].ravel())
        ds.close()

if not member_rows:
    raise RuntimeError('No randomization_experiment_*.nc files found -- run 07_randomization.sh first.')

member_table = pd.DataFrame(member_rows).set_index('member')
print(member_table.round(3).to_string())

In [ ]:
orig_all = np.concatenate(orig_pooled)
null_all = np.concatenate(null_pooled)

n_members = len(member_table)
uniq_basins = sorted(set(basin_counts.values()))

print('=' * 70)
print('FIGURE 5a POOLED STATISTICS')
print('=' * 70)
print(f'Ensemble members (precip x SST files found) : {n_members}   <-- YY')
print(f'Basins per member                           : {uniq_basins}   <-- XX')
if len(uniq_basins) > 1:
    print('  WARNING: members disagree on basin count; XX is ambiguous.')
    print(f'  {basin_counts}')
print()
print(f'Pooled observed correlations n = {orig_all.size:,}')
print(f'  mean   = {orig_all.mean():.4f}   <-- claim says 0.27')
print(f'  median = {np.median(orig_all):.4f}')
print(f'  positive: {(orig_all > 0).sum():,} of {orig_all.size:,} '
      f'({(orig_all > 0).mean() * 100:.2f}%)   <-- supports "nearly all"')
print()
print(f'Pooled null correlations n = {null_all.size:,}')
print(f'  mean   = {null_all.mean():.4f}   <-- claim says centered on zero')
print(f'  median = {np.median(null_all):.4f}')
print(f'  std    = {null_all.std():.4f}')
print(f'  positive: {(null_all > 0).mean() * 100:.2f}%')
print()
print(f'Fraction of null values >= pooled observed mean : '
      f'{np.mean(null_all >= orig_all.mean()):.4f}')
print(f'Mean % of basins per member with empirical p < 0.05 : '
      f'{member_table["pct_p_lt_05"].mean():.1f}%')
print()
print('Basins positive in EVERY member (strictest reading of "nearly all"):')
sizes = {a.size for a in orig_raw.values()}
if len(sizes) == 1:
    stacked = np.vstack(list(orig_raw.values()))          # (member, basin), NaNs kept
    all_pos = np.all(stacked > 0, axis=0)                 # NaN > 0 is False, so NaN excludes a basin
    complete = np.all(np.isfinite(stacked), axis=0)
    print(f'  {all_pos.sum()} of {all_pos.size} basins '
          f'({all_pos.mean() * 100:.1f}%) positive in all {stacked.shape[0]} members')
    print(f'  ({complete.sum()} basins have a finite value in every member)')
else:
    print(f'  skipped -- basin counts differ across members: {sorted(sizes)}')

---
## 3. Stability panel: how many 30-year windows, and over what period?

From `15_Stability_of_the_marginal_sensitivity.py`:

```python
WINDOW_SIZE = 30   # years  -> 360 months
WINDOW_STEP = 12   # months
n_windows = (n_times - window_months) // WINDOW_STEP + 1
```

`n_times` is the length of the common time axis of the **first** precip dataset and the
**first** SST dataset only (`15_...py:141-149`). No years are deliberately dropped --
the count follows entirely from the record length:

| record | months | windows | last-window end years |
|---|---|---|---|
| 1980-01 to 2024-12 | 540 | 16 | 2009-2024 |
| 1980-01 to 2023-12 | 528 | 15 | 2009-2023 |
| 1980-01 to 2022-12 | 516 | 14 | 2009-2022 |

So **14 windows implies the common record ends in 2022**, which contradicts a stated
period of 1980-2024. The cells below establish which is true from the data itself.

In [ ]:
# What the saved output actually contains
pc_path = DATA_DIR / 'pattern_correlations_all_basins.nc'

if pc_path.exists():
    pc = xr.open_dataset(pc_path)
    corr = pc['pattern_corr_per_member']
    windows = pc['window'].values

    print('Attributes written by script 15:')
    for k, v in pc.attrs.items():
        print(f'  {k:22s}: {v}')
    print()
    print('Dimensions :', dict(corr.sizes))
    print(f'n_windows  : {len(windows)}   <-- the claimed 14')
    print(f'window end years: {list(windows)}')
    print(f'  first window covers {windows[0] - 29}-01 to {windows[0]}-12')
    print(f'  last  window covers {windows[-1] - 29}-01 to {windows[-1]}-12')
    print(f'  => implied full record: {windows[0] - 29} to {windows[-1]}')
    print()
    print(f'n_basins   : {corr.sizes.get("basin", "n/a")}')
    print(f'n_ensemble : {corr.sizes.get("ensemble", "n/a")}   <-- members in panel b')
    if 'ensemble' in corr.coords:
        print(f'members    : {list(corr["ensemble"].values)}')
    print()
    print(f'Pattern correlation overall mean = {float(corr.mean()):.3f}')
    print(f'  min = {float(corr.min()):.3f}, max = {float(corr.max()):.3f}')
else:
    print(f'[missing] {pc_path} -- run 09_stability.sh first.')

In [ ]:
# Reproduce the window arithmetic from the raw anomaly files.
# NOTE: do NOT import 15_Stability_of_the_marginal_sensitivity.py -- it is top-level
# code with no main() guard and would rerun the whole analysis on import.

WINDOW_SIZE, WINDOW_STEP = 30, 12
window_months = WINDOW_SIZE * 12

# USE_LONGER_PERIOD = True branch of script 15
PRECIP_15 = ['GPCP', 'CRU', 'GPCC', 'CPC', 'PREC', 'TerraClimate']
SST_15 = ['ERSSTv6', 'COBE-SST3']

sst_times = {}
for s in SST_15:
    p = INPUTS_DIR / f'sst_anom_{s}.nc'
    if p.exists():
        sst_times[s] = xr.open_dataarray(p).squeeze()['time'].values

precip_times = {}
for p_name in PRECIP_15:
    p = INPUTS_DIR / f'precip_anom_{p_name}.nc'
    if p.exists():
        precip_times[p_name] = xr.open_dataarray(p)['time'].values

if not sst_times or not precip_times:
    raise RuntimeError(f'No anomaly inputs found under {INPUTS_DIR}')

print('Record coverage of each input:')
for name, t in {**precip_times, **sst_times}.items():
    print(f'  {name:14s} {str(t[0])[:7]} to {str(t[-1])[:7]}  ({len(t)} months)')
print()

rows = []
for s_name, s_t in sst_times.items():
    for p_name, p_t in precip_times.items():
        ct = np.intersect1d(p_t, s_t)
        n_win = (len(ct) - window_months) // WINDOW_STEP + 1 if len(ct) >= window_months else 0
        rows.append({
            'pair': f'{p_name}_{s_name}',
            'start': str(ct[0])[:7] if len(ct) else 'n/a',
            'end': str(ct[-1])[:7] if len(ct) else 'n/a',
            'n_months': len(ct),
            'n_windows_if_first': n_win,
        })

pair_table = pd.DataFrame(rows).set_index('pair')
print(pair_table.to_string())

In [ ]:
# Script 15 takes n_times from the FIRST precip x FIRST SST pair only, then applies
# those integer window indices positionally to every other pair. If the pairs do not
# share an identical time axis, the window labels do not describe every member.

first_p = list(precip_times)[0]
first_s = list(sst_times)[0]
ct_first = np.intersect1d(precip_times[first_p], sst_times[first_s])
n_times = len(ct_first)
n_windows_calc = (n_times - window_months) // WINDOW_STEP + 1

print(f'Reference pair used by script 15 : {first_p} x {first_s}')
print(f'  common record : {str(ct_first[0])[:7]} to {str(ct_first[-1])[:7]}  ({n_times} months)')
print(f'  n_windows     = ({n_times} - {window_months}) // {WINDOW_STEP} + 1 = {n_windows_calc}')
print()

end_years = [int(str(ct_first[i * WINDOW_STEP + window_months - 1])[:4])
             for i in range(n_windows_calc)]
print(f'  window end years: {end_years}')
print(f'  => x-axis spans {end_years[0]} to {end_years[-1]}')
print()

lengths = pair_table['n_months'].unique()
if len(lengths) == 1:
    print('All pairs share the same record length -- positional windows are consistent.')
else:
    print('WARNING: pairs have DIFFERENT record lengths:', sorted(lengths))
    print('Script 15 slices every pair with indices derived from the reference pair,')
    print('so for shorter/offset members the windows cover different actual months than')
    print('the labels imply. Members shorter than the reference silently yield short windows.')
    short = pair_table[pair_table['n_months'] != n_times]
    print()
    print(short.to_string())

print()
print('Members with a full 1980-2024 record (the phrase "at least 1980-2024"):')
full = pair_table[(pair_table['start'] <= '1980-01') & (pair_table['end'] >= '2024-12')]
print(f'  {len(full)} of {len(pair_table)} pairs qualify')
if len(full):
    print(full.to_string())

---
## 4. Filled-in sentences

Run after all sections above. Substitute the printed values into the manuscript.

In [ ]:
print('=' * 78)
print('SENTENCES WITH NUMBERS SUBSTITUTED')
print('=' * 78)
print()

if var_tables:
    for name, tbl in var_tables.items():
        pct = tbl.loc[15, 'cumulative_pct']
        print(f'[1] ({name}) ... retaining the first 15 modes, which explain '
              f'{pct:.0f}% of the total ocean variance.')
    print('    NOTE: K_MAX=15 is the max k TESTED in CV; the fitted model uses the')
    print('    1-SE-rule optimal_k per basin. Check section 1b before using this wording.')
    print()

if len(member_table):
    xx = uniq_basins[0] if len(uniq_basins) == 1 else '??'
    print(f'[2] In nearly all {xx} river basins and {n_members} ensemble members the')
    print(f'    correlation is positive ({(orig_all > 0).mean() * 100:.1f}% of '
          f'{orig_all.size:,} basin-member pairs), with a mean correlation of')
    print(f'    {orig_all.mean():.2f}, whereas the null distribution is centered on')
    print(f'    {null_all.mean():.3f}.')
    print()

if pc_path.exists():
    print(f'[3] Stability across {corr.sizes.get("basin", "?")} basins and '
          f'{corr.sizes.get("ensemble", "?")} ensemble members;')
    print(f'    {len(windows)} 30-year windows ending {windows[0]}-{windows[-1]},')
    print(f'    i.e. a full record of {windows[0] - 29}-{windows[-1]}.')
    print(f'    Stated period in file attrs: {pc.attrs.get("time_period", "n/a")}')
    if int(windows[0]) - 29 != 1980 or int(windows[-1]) != 2024:
        print('    MISMATCH: window span does not equal 1980-2024. Update the text to')
        print(f'    say {windows[0] - 29}-{windows[-1]}, or rerun once longer data exist.')
    print('    No years are dropped -- the count is (n_months - 360) // 12 + 1.')